# OpenPlaque — Left-Coronary Local Root-Directed Bridge v1.1

Connectivity-fix rerun of the same experiment. Deep anatomy is excluded without artificially sealing epicardial boundary neighborhoods, and disconnected variants are recorded as failed gates instead of crashing. **Runtime → Run all**.


In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)


In [ ]:
from pathlib import Path
import json, os, shutil, sys, time
MOUNT = Path('/content/drive')
MARKER_REL = Path('Cache/Secondary_3D_Vesselness_Topology_v1/series7_int16.npy')
MASTER_REL = Path('Cache/Master_Coronary_Anatomy_Baseline_v2/master_anatomy_summary.json')
candidate_roots = [Path('/content/drive/MyDrive/OpenPlaque')]
mydrive = Path('/content/drive/MyDrive')
if mydrive.exists():
    candidate_roots += [p for p in mydrive.iterdir() if p.is_dir() and p.name != 'OpenPlaque']
shared = Path('/content/drive/Shareddrives')
if shared.exists():
    for sd in shared.iterdir():
        if sd.is_dir(): candidate_roots += [sd/'OpenPlaque', sd]
DRIVE_ROOT = next((p for p in candidate_roots if (p/MARKER_REL).exists() and (p/MASTER_REL).exists()), None)
if DRIVE_ROOT is None:
    top = sorted(p.name for p in mydrive.iterdir())[:80] if mydrive.exists() else []
    raise RuntimeError('Mounted Google Drive does not contain the established OpenPlaque cache/master. Remount the correct Google account and use Runtime → Run all.\nTop-level MyDrive entries: '+str(top))
OUTPUT = DRIVE_ROOT / 'Left_Coronary_Local_Root_Directed_Bridge_v1'
REUSE_VALID_CACHES = True
FORCE_RECOMPUTE = True  # clean rerun after connectivity bugfix
if FORCE_RECOMPUTE and OUTPUT.exists(): shutil.rmtree(OUTPUT)
OUTPUT.mkdir(parents=True, exist_ok=True)
(OUTPUT/'notebook_started.json').write_text(json.dumps({'status':'started','time':time.time(),'drive_root':str(DRIVE_ROOT)}, indent=2))
print('Validated Drive root:', DRIVE_ROOT)
print('Output:', OUTPUT)


In [ ]:
import os, shutil, sys
os.chdir('/content')
REPO = Path('/content/OpenPlaque_left_local_root')
if REPO.exists(): shutil.rmtree(REPO)
BRANCH = 'left-coronary-local-root-directed-bridge-from-main'
PINNED_SCIENCE_COMMIT = '1744c558f5e9d9159890f70880fc63be6d183d92'
BASELINE = '0593b453959f5a353d644267fbeef24b514ef4d7'
!git clone -q --branch $BRANCH https://github.com/pazzani/OpenPlaque.git $REPO
!git -C $REPO checkout -q $PINNED_SCIENCE_COMMIT
HEAD = get_ipython().getoutput(f'git -C {REPO} rev-parse HEAD')[0].strip()
MB = get_ipython().getoutput(f'git -C {REPO} merge-base HEAD {BASELINE}')[0].strip()
print('Checked out:', HEAD)
print('Merge base:', MB)
assert HEAD == PINNED_SCIENCE_COMMIT
assert MB == BASELINE
%pip install -q SimpleITK scipy matplotlib pandas scikit-image
%pip install -q /content/OpenPlaque_left_local_root
for k in list(sys.modules):
    if k == 'openplaque' or k.startswith('openplaque.'):
        del sys.modules[k]
os.chdir('/content')


In [ ]:
from openplaque.left_coronary_local_root_directed_bridge_v1 import synthetic_self_test
print('Synthetic self-test:', synthetic_self_test())
!pytest -q /content/OpenPlaque_left_local_root/tests/test_left_coronary_local_root_directed_bridge_v1.py


In [ ]:
required = [
    DRIVE_ROOT/'Cache/Secondary_3D_Vesselness_Topology_v1/series7_int16.npy',
    DRIVE_ROOT/'Cache/Secondary_3D_Vesselness_Topology_v1/series7_int16.json',
    DRIVE_ROOT/'Cache/Master_Coronary_Anatomy_Baseline_v2/master_anatomy_summary.json',
    DRIVE_ROOT/'Cache/LAD_Frozen_Proximal_Reacquisition_v1/combined_lad_centerline.csv',
    DRIVE_ROOT/'Cache/Source_Volume_Coronary_Centerlines/RCA_source_centerline.csv',
    DRIVE_ROOT/'TotalSegmentator_Cardiovascular_Cache_v1/heartchambers_highres/aorta.nii.gz',
    DRIVE_ROOT/'TotalSegmentator_Cardiovascular_Cache_v1/heartchambers_highres/pulmonary_artery.nii.gz',
    DRIVE_ROOT/'TotalSegmentator_Cardiovascular_Cache_v1/heartchambers_highres/heart_atrium_left.nii.gz',
    DRIVE_ROOT/'TotalSegmentator_Cardiovascular_Cache_v1/heartchambers_highres/heart_atrium_right.nii.gz',
    DRIVE_ROOT/'TotalSegmentator_Cardiovascular_Cache_v1/heartchambers_highres/heart_ventricle_left.nii.gz',
    DRIVE_ROOT/'TotalSegmentator_Cardiovascular_Cache_v1/heartchambers_highres/heart_ventricle_right.nii.gz',
]
missing = [str(p) for p in required if not p.exists()]
if missing: raise FileNotFoundError('Missing prerequisites:\n'+'\n'.join(missing))
(OUTPUT/'preflight_complete.json').write_text(json.dumps({'status':'complete','science_commit':PINNED_SCIENCE_COMMIT,'baseline':BASELINE,'required_count':len(required)}, indent=2))
print('Preflight complete:', len(required), 'required artifacts found')


In [ ]:
from openplaque.left_coronary_local_root_directed_bridge_v1 import run
try:
    result = run(drive_root=str(DRIVE_ROOT), output_dir=str(OUTPUT))
    print(json.dumps(result['summary'], indent=2, default=str))
    print('Report:', result['report'])
    print('ZIP:', result['zip'])
except Exception as e:
    (OUTPUT/'notebook_failure.json').write_text(json.dumps({'status':'FAILED','type':type(e).__name__,'message':str(e)}, indent=2))
    raise
